# Libraries Import

In [1]:
import matplotlib.pyplot as plt
plt.style.use("seaborn")
# import MetaTrader5 as mt5
import plotly.express as px
from prepare_data import *
from Strategies.Stratey_bb_adx_rsi_ema200_1MIN import Strategy_bb_adx_rsi_ema200_1MIN
from Strategies.Strategy_rsi_stoch_ema50 import Strategy_rsi_stoch_ema50

C:\Users\mikep\AppData\Local\Temp\ipykernel_13916\984952790.py:2: MatplotlibDeprecationWarning: The seaborn styles shipped by Matplotlib are deprecated since 3.6, as they no longer correspond to the styles shipped by seaborn. However, they will remain available as 'seaborn-v0_8-<style>'. Alternatively, directly use the seaborn API instead.
  plt.style.use("seaborn")


# Import Data

In [2]:
data = import_csv_data("./csv_files/1H/US100_1H.csv")
data = clean_data(data)


# Prepare Data

In [3]:
data = prepare_data(data)

c:\Users\mikep\AppData\Local\Programs\Python\Python311\Lib\site-packages\ta\trend.py:730: FutureWarning:

The behavior of `series[i:j]` with an integer-dtype index is deprecated. In a future version, this will be treated as *label-based* indexing, consistent with e.g. `series[i]` lookups. To retain the old behavior, use `series.iloc[i:j]`. To get the future behavior, use `series.loc[i:j]`.

c:\Users\mikep\AppData\Local\Programs\Python\Python311\Lib\site-packages\ta\trend.py:780: RuntimeWarning:

invalid value encountered in double_scalars

c:\Users\mikep\AppData\Local\Programs\Python\Python311\Lib\site-packages\ta\trend.py:785: RuntimeWarning:

invalid value encountered in double_scalars



In [17]:
data[data["MCR"].notnull()]["MCR"]

time
2022-03-18 14:00:00+01:00             (bearish, 14093.554)
2022-03-21 16:00:00+01:00    (bearish, 14408.730000000001)
2022-03-21 17:00:00+01:00             (bullish, 14326.776)
2022-03-21 20:00:00+01:00             (bearish, 14329.144)
2022-03-22 08:00:00+01:00             (bearish, 14371.042)
                                         ...              
2022-11-22 15:00:00+01:00              (bullish, 11539.44)
2022-11-22 16:00:00+01:00              (bearish, 11581.96)
2022-11-22 21:00:00+01:00              (bearish, 11707.55)
2022-11-23 15:00:00+01:00             (bearish, 11783.044)
2022-11-25 10:00:00+01:00              (bullish, 11838.29)
Name: MCR, Length: 374, dtype: object

# Parameters

In [ ]:
ratio = 1.01
security_sl = 10
security_sl_1 = security_sl
security_sl_2 = security_sl
volume_control = 0
starting_balance = 1000
volume = 1
spread = 1*volume

# Run the Strategy

In [ ]:
rsi_stoch_strategy = Strategy_rsi_stoch_ema50(data, starting_balance=starting_balance, volume=volume, security_sl=security_sl, ratio=ratio, spread=spread)
result = rsi_stoch_strategy.run()

### Plot result

In [ ]:
plot_trades_result(data, result)

In [ ]:
px.line(result, x='close_datetime', y='pnl')

### % win vs % loss

In [ ]:
loss = 0
win = 0
for index, row in result.iterrows():
  if row["profit"] < 0:
    loss = loss + 1
  else :
    win = win + 1

Win_ = (win) / (win + loss) * 100
Loss_ = loss / (win+loss) * 100


In [ ]:
print(f"pourcentage Win {Win_}%")

In [ ]:
print(f"pourcentage Loss {Loss_}%")

# Optimization

### Optimization SL

In [ ]:
starting_balance = 1000
volume = 1
spread = 1*volume
ratio = 1.3

max_profit = {"security_sl": 0, "pnl": 0}

for i in range(300):
  rsi_stoch_strategy = Strategy_rsi_stoch_ema50(data, starting_balance=starting_balance, volume=volume, security_sl=i, ratio=ratio, spread=spread)
  result = rsi_stoch_strategy.run()
  current_max_profit = result["pnl"].values[-2]
  if current_max_profit > max_profit["pnl"]:
    max_profit["security_sl"] = i
    max_profit["pnl"] = current_max_profit

max_profit


# Optimization +++

#### Optimization SL ++

In [ ]:
starting_balance = 1000
volume = 1
spread = 1*volume

max_profit = {"security_sl": 0, "ratio": 0, "pnl": 0}

for i in range(100, 300):
  ratio = i/100
  for j in range(100):
    rsi_stoch_strategy = Strategy_rsi_stoch_ema50(data, starting_balance=starting_balance, volume=volume, ratio=ratio, spread=spread, security_sl=j)
    result = rsi_stoch_strategy.run()
    current_max_pnl = result["pnl"].iat[-1]
    if current_max_pnl > max_profit["pnl"]:
      max_profit["security_sl"] = j
      max_profit["ratio"] = ratio
      max_profit["pnl"] = current_max_pnl

max_profit
    

max_profit

# Excel File

In [ ]:
result.drop(result.tail(1).index,inplace=True)
write_excel_file(result)

# Saved Results

### US100-1H
#### Double sl : 1) 47.5 < atr < 62.5, ratio=1.5
{'security_sl_1': 15, 'security_sl_2': 22, 'pnl': 4429}

### US100-15MIN
#### Simple sl : sl = 5, ratio = 2.49
{'security_sl': 5, 'ratio': 2.49, 'pnl': 5225}

# Optimization Max Drawdown

In [ ]:
starting_balance = 1000
volume = 1
spread = 1*volume
ratio = 1.5

max_profit = {"security_sl": 0, "pnl": 0, "mdd": -1000}

for i in range(300):
  rsi_stoch_strategy = Strategy_rsi_stoch_ema50(data, starting_balance=starting_balance, volume=volume, security_sl=i, ratio=ratio, spread=spread)
  result = rsi_stoch_strategy.run()
  current_max_dd = max_drawdown(result["pnl"])
  current_max_profit = result["pnl"].iat[-1]
  if current_max_dd > max_profit["mdd"]:
    max_profit["security_sl"] = i
    max_profit["pnl"] = current_max_profit
    max_profit['mdd'] = current_max_dd

max_profit